#### 문서의 내용을 읽고 쪼개기

In [ ]:
%pip install docx2txt
%pip install langchain-community docx2txt
%pip install -qU langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size= 1500,       ## 하나의 청크가 가질 토큰 수(청크 크기)
  chunk_overlap= 200      ## 청크 간에 중복시킬 토큰 수
)

loader= Docx2txtLoader('Tax.docx')


document_list  = loader.load_and_split(text_splitter= splitter)

document_list

#### 문서 임베딩 후 벡터 데이터베이스로 저장

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
%pip install langchain-upstage

In [3]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model = 'embedding-passage')

#### ChromaDB로 임베디드 DB 만들기

In [ ]:
%pip install langchain-chroma

In [4]:
from langchain_chroma import Chroma

# 처음 DB 생성할 때는 이 방식으로

# database = Chroma.from_documents(
#   documents= document_list,
#   embedding=embedding, 
#   collection_name='chroma-tax',
#   persist_directory= 'upstage_chroma'
# )

database = Chroma(
  collection_name= 'chroma-tax',
  persist_directory= './upstage_chroma',
  embedding_function= embedding     # 요거 안쓰면 Chroma 기본 차원수 384
)

#### Retrieve

In [22]:
query = '기타소득의 세율을 기타소득의 종류별로 설명해주세요'

retrieved_docs = database.similarity_search(query=query, k=20)

In [ ]:
retrieved_docs

#### Augmented Generation

In [ ]:
%pip install langchain-anthropic

In [9]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
  model = 'claude-3-haiku-20240307',
  temperature = 0.3,    # 높을수록 창의적, 랜덤
  top_p = 1             # 낮을수록 확실한 토큰만 선택
)

In [10]:
prompt = f'''[Identity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [Context]를 참고하여 사용자의 [Question]에 답변해주세요.

[Context]
{retrieved_docs}

[Question]
{query}
'''

In [11]:
ai_message= llm.invoke(prompt)

In [12]:
ai_message.content

'기타소득에 대한 세율은 다음과 같습니다:\n\n1. 제21조제1항제1호부터 제8호까지, 제8호의2, 제9호부터 제20호까지, 제22호, 제22호의2 및 제26호에 따른 기타소득 중 기타소득금액이 300만원 이하인 경우: 100분의 15\n2. 제21조제1항제2호에 따른 기타소득 중 복권 당첨금: 100분의 20\n3. 그 밖의 기타소득: 100분의 20\n\n다만, 제21조제1항제8호라목 및 마목에 해당하는 소득금액이 3억원을 초과하는 경우 그 초과하는 부분에 대해서는 100분의 30의 세율이 적용됩니다.\n\n또한 제21조제1항제21호에 따른 연금외수령한 기타소득과 제21조제1항제27호 및 같은 조 제2항에 따른 기타소득에 대해서는 각각 다른 세율이 적용됩니다.\n\n종합적으로 기타소득의 종류와 금액에 따라 세율이 달리 적용되는 것을 알 수 있습니다.'

#### LCEL

In [ ]:
%pip install -U langchain langchain-community langchain-core

In [23]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 context를 바탕으로 질문에 답하세요.
     당신은 K-IFRS 회계 기준 전문가입니다. 다음 Context를 바탕으로 사용자의 질문에 대해 명확하고 정확하게 답변해주세요.
     
     [답변 작성 규칙]
     1. 질문에 대한 핵심 내용만 간결하게 요약해서 답변하세요.
     2. 불필요한 배경 설명(예: 도입 배경, US GAAP 비교, 라이선스 세부 사례 등)은 제외하세요.
     3. n단계 모형을 설명할 때는 1단계부터 n단계까지 번호를 매겨서 명확히 구분하세요.
     4. 각 단계 설명은 1~2문장으로 짧게 요약하세요.
     5. 문서를 꼼꼼히 확인하고 사실과 다른 내용은 지어내지 마세요.
     
     다음은 Few-shot 예시입니다. 
     question: 확신유형의 보증과 용역유형의 보증에 대해 설명해줘.
     answer: 확신유형의 보증은 수행의무로 회계처리하지 않고 용역유형의 보증은 수행의무로 회계처리한다.
     
     Let's think step by step
     
     \n\nContext: {context}"""),
    ("human", "{question}")
])

# 문서 포맷팅 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 체인 구성
qa_chain = (
    {
        "context": database.as_retriever(k=20) | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 실행
answer = qa_chain.invoke(query)
print(answer)

기타소득의 세율은 다음과 같이 종류별로 구분됩니다.

1. 상금, 현상금, 포상금, 보로금 등 - 20% 세율 적용
2. 복권, 경품권 등 추첨권 당첨금 - 20% 세율 적용 
3. 사행행위 참가로 얻은 재산상 이익 - 20% 세율 적용
4. 방송 해설, 계몽, 연기 심사 등 용역 소득 - 20% 세율 적용
5. 전문직 용역 소득(변호사, 회계사 등) - 20% 세율 적용
6. 그 외 고용관계 없이 수당 등을 받는 용역 소득 - 20% 세율 적용
7. 법인세법상 기타소득으로 처분된 소득 - 20% 세율 적용
8. 연금계좌 연금외수령 소득 - 20% 세율 적용
9. 주식매수선택권 행사 이익 - 20% 세율 적용
10. 퇴직 후 직무발명보상금 - 20% 세율 적용
11. 뇌물, 알선수재, 배임수재 금품 - 30% 세율 적용
12. 종교관련종사자 종교인소득 - 기본세율 적용

이와 같이 기타소득의 종류에 따라 세율이 20%, 30% 등으로 다양하게 적용됩니다.


In [19]:
query = '사업소득이 있는 자가 받을 수 있는 세액공제에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

사업소득이 있는 자가 받을 수 있는 세액공제는 다음과 같습니다:

1. 표준세액공제: 
   - 종합소득이 있는 거주자(근로소득이 있는 자는 제외)로서 세액공제 신청을 하지 않은 경우
   - 성실사업자의 경우 연 12만원, 그 외의 경우 연 7만원을 공제받을 수 있음

2. 특별세액공제:
   - 보험료공제, 의료비공제, 교육비공제, 기부금공제 등 다양한 항목에 대해 공제받을 수 있음
   - 이 경우 해당 거주자가 대통령령으로 정하는 바에 따라 신청한 경우에 적용됨

3. 기타 세액공제:
   - 외국납부세액공제, 재해손실세액공제 등 특별한 경우에 적용되는 세액공제도 있음

요약하면, 사업소득이 있는 자는 표준세액공제와 특별세액공제를 통해 세액을 절감할 수 있으며, 추가로 특별한 경우에 적용되는 세액공제도 활용할 수 있습니다.


In [20]:
query = '2. 전자계산서 발급 세액공제(제56조의3)의 요건에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

전자계산서 발급 세액공제(제56조의3)의 요건은 다음과 같습니다:

1. 전자계산서 발급 요건
- 「부가가치세법」 제32조제2항에 따른 전자세금계산서를 발급하여야 하는 사업자
- 「조세특례제한법」 제63조의3제1항에 따른 전자계산서를 발급한 사업자

2. 세액공제 요건
- 전자계산서를 발급한 경우
- 전자계산서 발급명세를 국세청장에게 전송한 경우

3. 세액공제 금액
- 전자계산서 발급 건당 200원
- 연간 공제한도: 1천만원

즉, 전자계산서 발급 요건을 충족하고 발급명세를 국세청에 전송한 경우 전자계산서 발급 건당 200원씩 세액공제를 받을 수 있으며, 연간 공제한도는 1천만원입니다.


In [21]:
query = '과세표준이 5000만원인 직장인의 산출세액을 알려주세요. 단, 현행 세법은 누진세율임에 유의하시오.'
answer = qa_chain.invoke(query)
print(answer)

# 하아..

네, 알겠습니다. 과세표준이 5,000만원인 직장인의 산출세액은 다음과 같이 계산됩니다.

현행 소득세법상 누진세율 적용:

1. 과세표준 5,000만원
2. 세율 적용:
   - 1,200만원까지 6% 
   - 1,200만원 초과 4,600만원까지 15%
   - 4,600만원 초과 부분 24%
3. 산출세액 계산:
   - 1,200만원까지: 1,200만원 x 6% = 72만원
   - 1,200만원 초과 4,600만원까지: (4,600만원 - 1,200만원) x 15% = 510만원 
   - 4,600만원 초과 부분: (5,000만원 - 4,600만원) x 24% = 96만원
4. 총 산출세액: 72만원 + 510만원 + 96만원 = 678만원

따라서 과세표준이 5,000만원인 직장인의 산출세액은 678만원입니다.


In [28]:
query = '간편장부대상자는 소득 증빙을 어떻게 해?'
answer = qa_chain.invoke(query)
print(answer)

간편장부대상자의 소득 증빙 방법은 다음과 같습니다:

1. 인적공제, 연금보험료공제 등 공제 대상임을 증명하는 서류 제출
2. 총수입금액과 필요경비 계산에 필요한 서류 제출
3. 제160조제2항에 따라 기장(記帳)을 한 사업자의 경우 재정경제부령으로 정하는 간편장부소득금액 계산서 제출
4. 제28조부터 제32조까지의 규정에 따라 필요경비를 산입한 경우 그 명세서 제출
5. 소규모사업자는 영수증 수취명세서 제출 의무 제외

즉, 간편장부대상자는 복식부기의무자에 비해 간소화된 증빙 서류를 제출하면 됩니다. 다만 성실하게 거래 사실을 기재한 간편장부를 비치해야 합니다.
